# 19 — Plan B: Independent Validation + Decision Engine

## Objective

Notebook 19 validates the operational Plan B decision engine on the existing bridge population.

The purpose is **not** to prove that one Bauwerksart is universally best.

The purpose is to test whether the transparent reference-cohort logic behaves consistently when existing bridges are used as pseudo-proposed bridges.

## Architecture

```text
Existing bridge held out as pseudo-input
        ↓
Plan B similarity engine
        ↓
Comparable reference cohort
        ↓
Performance / condition evidence
        ↓
Bauwerksart-level decision scores
        ↓
Validation against the held-out bridge's actual Bauwerksart
```

The frozen 86-predictor condition model is not changed.

No FEM design is performed.

## 01 — Current project inputs

Notebook 19 uses only the outputs already created by Notebooks 17 and 18.

```text
C:\Datenanalyse\final Project\Output_PlanA-B
```

Inputs:

```text
17_Plan_A_Germany_Web_Map/
    plan_a_map_data.parquet

18_Plan_B_Reference_Library/
    plan_b_reference_library.parquet
```

Outputs:

```text
19_Plan_B_Independent_Validation/
    plan_b_validation_results.parquet
    plan_b_validation_results.csv
    plan_b_type_validation_summary.csv
    19_plan_b_validation_manifest.json
```

In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Datenanalyse\final Project")
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

MAP17_INPUT = OUTPUT_ROOT / "17_Plan_A_Germany_Web_Map" / "plan_a_map_data.parquet"
LIBRARY18_INPUT = OUTPUT_ROOT / "18_Plan_B_Reference_Library" / "plan_b_reference_library.parquet"

OUTPUT_DIR = OUTPUT_ROOT / "19_Plan_B_Independent_Validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_PARQUET = OUTPUT_DIR / "plan_b_validation_results.parquet"
RESULT_CSV = OUTPUT_DIR / "plan_b_validation_results.csv"
TYPE_SUMMARY_CSV = OUTPUT_DIR / "plan_b_type_validation_summary.csv"
MANIFEST_JSON = OUTPUT_DIR / "19_plan_b_validation_manifest.json"

for p in [MAP17_INPUT, LIBRARY18_INPUT]:
    if not p.exists():
        raise FileNotFoundError(f"Required current-project input not found:\n{p}")

print("[PASS] Notebook 19 input paths")

[PASS] Notebook 19 input paths


In [2]:
map17 = pd.read_parquet(MAP17_INPUT)
library = pd.read_parquet(LIBRARY18_INPUT)

for name, df in [("map17", map17), ("library", library)]:
    if "bridge_id" not in df.columns:
        raise KeyError(f"{name} missing bridge_id")
    df["bridge_id"] = df["bridge_id"].astype("string").str.strip()
    if not df["bridge_id"].is_unique:
        raise ValueError(f"{name} has duplicate bridge_id")

if len(map17) != 52214:
    raise ValueError(f"Notebook 17 population must be 52,214; found {len(map17)}")

if len(library) != 52214:
    raise ValueError(f"Notebook 18 library must contain 52,214; found {len(library)}")

required = [
    "latitude","longitude","bauwerksart_text","baustoffklasse",
    "laenge","breite","dtv_reference","performance_score"
]
missing = [c for c in required if c not in library.columns]
if missing:
    raise KeyError(f"Reference library missing required fields: {missing}")

print("[PASS] reference library loaded")

[PASS] reference library loaded


## 02 — Validation design

A random sample of existing bridges is treated as pseudo-new bridges.

For each held-out bridge:

- its own record is excluded from the reference cohort;
- its location, geometry, DTV and material are used as the proposed-bridge input;
- **Bauwerksart is withheld from the decision engine**;
- the engine returns a type-level ranking;
- the actual held-out Bauwerksart is then compared with the result.

This avoids the most important leakage: a bridge cannot recommend itself.

The validation is a **decision-engine consistency test**, not an independent test of the frozen condition model.

In [3]:
RANDOM_SEED = 20260920
VALIDATION_SAMPLE_SIZE = 1000
TOP_K_REFERENCES = 25
MIN_REFERENCES_PER_TYPE = 5

rng = np.random.default_rng(RANDOM_SEED)

eligible = library[
    library["latitude"].notna()
    & library["longitude"].notna()
    & library["laenge"].notna()
    & library["breite"].notna()
    & library["bauwerksart_text"].notna()
    & library["baustoffklasse"].notna()
].copy()

if len(eligible) < VALIDATION_SAMPLE_SIZE:
    raise ValueError(
        f"Not enough eligible bridges for validation: {len(eligible)}"
    )

validation_ids = (
    eligible["bridge_id"]
    .sample(n=VALIDATION_SAMPLE_SIZE, random_state=RANDOM_SEED)
    .tolist()
)

print("Eligible validation population:", len(eligible))
print("Validation sample:", len(validation_ids))
print("Seed:", RANDOM_SEED)
print("[PASS] validation cohort created")

Eligible validation population: 51427
Validation sample: 1000
Seed: 20260920
[PASS] validation cohort created


## 03 — Similarity configuration

The validation uses the exact transparent initial weights recorded by Notebook 18:

```text
Bauwerksart   30%
Baustoff      20%
Länge         15%
Breite        10%
DTV           15%
Distanz       10%
```

During validation, `Bauwerksart` is deliberately omitted from the input because it is the target being tested.

Therefore the available weights are renormalized over:

```text
Baustoff + Länge + Breite + DTV + Distanz
```

The held-out bridge's Bauwerksart is used only after the engine has produced its ranking.

In [4]:
BASE_WEIGHTS = {
    "bauwerksart": 0.30,
    "baustoff": 0.20,
    "length": 0.15,
    "width": 0.10,
    "dtv": 0.15,
    "distance": 0.10,
}

ACTIVE_WEIGHTS = {
    k: v for k, v in BASE_WEIGHTS.items()
    if k != "bauwerksart"
}

weight_sum = sum(ACTIVE_WEIGHTS.values())
ACTIVE_WEIGHTS = {
    k: v / weight_sum for k, v in ACTIVE_WEIGHTS.items()
}

DISTANCE_SCALE_KM = 50.0
length_scale = max(float(library["laenge"].dropna().quantile(0.75)), 1.0)
width_scale = max(float(library["breite"].dropna().quantile(0.75)), 1.0)

print("Validation weights:", ACTIVE_WEIGHTS)
print("[PASS] validation weights")

Validation weights: {'baustoff': 0.28571428571428575, 'length': 0.2142857142857143, 'width': 0.14285714285714288, 'dtv': 0.2142857142857143, 'distance': 0.14285714285714288}
[PASS] validation weights


In [5]:
def haversine_km(lat, lon, ref_lat, ref_lon):
    lat1 = np.radians(float(lat))
    lon1 = np.radians(float(lon))
    lat2 = np.radians(np.asarray(ref_lat, dtype=float))
    lon2 = np.radians(np.asarray(ref_lon, dtype=float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat/2.0)**2
        + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    )
    return 6371.0088 * 2.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def cat_sim(series, value):
    return series.astype("string").eq(str(value)).astype(float)

def num_sim(series, value, scale):
    x = pd.to_numeric(series, errors="coerce")
    return pd.Series(
        np.where(x.notna(), np.exp(-np.abs(x-float(value))/scale), np.nan),
        index=series.index
    )

def dtv_sim(series, value):
    x = pd.to_numeric(series, errors="coerce")
    v = float(value)
    denom = np.maximum(np.maximum(np.abs(x), abs(v)), 1.0)
    score = np.clip(1.0 - np.abs(x-v)/denom, 0.0, 1.0)
    return pd.Series(np.where(x.notna(), score, np.nan), index=series.index)

print("[PASS] validation scoring functions")

[PASS] validation scoring functions


## 04 — Held-out scoring function

The reference bridge itself is removed before scoring.

The decision engine uses:

- material similarity;
- length similarity;
- width similarity;
- DTV similarity;
- geographic distance.

The type-level score is calculated from the top comparable reference bridges belonging to each Bauwerksart.

Performance evidence is included only after the reference cohort has been formed.

In [6]:
def validate_one(bridge_id):
    target = library.loc[library["bridge_id"] == bridge_id].iloc[0]

    refs = library[library["bridge_id"] != bridge_id].copy()

    refs["distance_km"] = haversine_km(
        target["latitude"],
        target["longitude"],
        refs["latitude"].to_numpy(),
        refs["longitude"].to_numpy(),
    )

    refs["sim_baustoff"] = cat_sim(
        refs["baustoffklasse"],
        target["baustoffklasse"]
    )
    refs["sim_length"] = num_sim(
        refs["laenge"],
        target["laenge"],
        length_scale
    )
    refs["sim_width"] = num_sim(
        refs["breite"],
        target["breite"],
        width_scale
    )
    refs["sim_dtv"] = dtv_sim(
        refs["dtv_reference"],
        target["dtv_reference"]
    )

    refs["sim_distance"] = np.exp(
        -refs["distance_km"] / DISTANCE_SCALE_KM
    )

    numerator = np.zeros(len(refs))
    denominator = np.zeros(len(refs))

    for name, col in [
        ("baustoff", "sim_baustoff"),
        ("length", "sim_length"),
        ("width", "sim_width"),
        ("dtv", "sim_dtv"),
        ("distance", "sim_distance"),
    ]:
        available = refs[col].notna().to_numpy()
        w = ACTIVE_WEIGHTS[name]
        numerator += np.where(
            available,
            refs[col].fillna(0).to_numpy() * w,
            0.0
        )
        denominator += np.where(available, w, 0.0)

    refs["similarity_score"] = np.where(
        denominator > 0,
        numerator / denominator,
        np.nan
    )

    refs = refs[refs["similarity_score"].notna()].copy()
    refs = refs.sort_values(
        ["similarity_score", "performance_score"],
        ascending=[False, False]
    )

    top_refs = refs.head(TOP_K_REFERENCES).copy()

    type_summary = (
        refs.groupby("bauwerksart_text", dropna=False)
        .agg(
            reference_count=("bridge_id", "count"),
            mean_similarity=("similarity_score", "mean"),
            mean_performance=("performance_score", "mean"),
            mean_distance_km=("distance_km", "mean"),
        )
        .reset_index()
    )

    type_summary["eligible"] = (
        type_summary["reference_count"] >= MIN_REFERENCES_PER_TYPE
    )

    type_summary["type_score"] = (
        0.80 * type_summary["mean_similarity"]
        + 0.20 * type_summary["mean_performance"].fillna(0.0)
    )

    type_summary = type_summary[
        type_summary["eligible"]
    ].sort_values(
        ["type_score", "reference_count"],
        ascending=[False, False]
    ).reset_index(drop=True)

    if type_summary.empty:
        return {
            "bridge_id": bridge_id,
            "actual_bauwerksart": target["bauwerksart_text"],
            "predicted_bauwerksart": pd.NA,
            "top1_correct": False,
            "reference_count_used": len(top_refs),
            "candidate_type_count": 0,
            "top1_score": np.nan,
            "actual_type_rank": np.nan,
            "actual_type_score": np.nan,
        }

    predicted = type_summary.iloc[0]["bauwerksart_text"]

    actual_rows = type_summary[
        type_summary["bauwerksart_text"].astype("string")
        == str(target["bauwerksart_text"])
    ]

    if actual_rows.empty:
        actual_rank = np.nan
        actual_score = np.nan
    else:
        actual_rank = int(actual_rows.index[0] + 1)
        actual_score = float(actual_rows.iloc[0]["type_score"])

    return {
        "bridge_id": bridge_id,
        "actual_bauwerksart": target["bauwerksart_text"],
        "predicted_bauwerksart": predicted,
        "top1_correct": str(predicted) == str(target["bauwerksart_text"]),
        "reference_count_used": len(top_refs),
        "candidate_type_count": len(type_summary),
        "top1_score": float(type_summary.iloc[0]["type_score"]),
        "actual_type_rank": actual_rank,
        "actual_type_score": actual_score,
    }

print("[PASS] held-out validation function")

[PASS] held-out validation function


## 05 — Run the independent validation sample

The sample is intentionally fixed by seed so the validation is reproducible.

The result is a descriptive validation of the decision engine's ability to retrieve the existing bridge type from comparable bridges.

In [7]:
rows = []

for i, bridge_id in enumerate(validation_ids, start=1):
    rows.append(validate_one(bridge_id))

validation_results = pd.DataFrame(rows)

if len(validation_results) != VALIDATION_SAMPLE_SIZE:
    raise RuntimeError("Validation result count mismatch.")

print("Validation rows:", len(validation_results))
print(
    "Top-1 type match rate:",
    validation_results["top1_correct"].mean()
)
print(
    "Actual type rank median:",
    validation_results["actual_type_rank"].median()
)

print("[PASS] validation run completed")

Validation rows: 1000
Top-1 type match rate: 0.313
Actual type rank median: 3.0
[PASS] validation run completed


## 06 — Type-level validation summary

The summary reports observed validation statistics.

No ranking is used as a project recommendation.

The purpose is to identify where the reference-cohort logic works, where the actual type is frequently absent from the eligible cohort, and where more evidence is needed.

In [8]:
type_summary = (
    validation_results
    .groupby("actual_bauwerksart", dropna=False)
    .agg(
        validation_count=("bridge_id", "count"),
        top1_match_rate=("top1_correct", "mean"),
        median_actual_type_rank=("actual_type_rank", "median"),
        mean_actual_type_score=("actual_type_score", "mean"),
    )
    .reset_index()
    .sort_values("validation_count", ascending=False)
)

overall = pd.DataFrame([{
    "validation_sample": len(validation_results),
    "top1_match_rate": validation_results["top1_correct"].mean(),
    "median_actual_type_rank": validation_results["actual_type_rank"].median(),
    "actual_type_available_rate": validation_results["actual_type_rank"].notna().mean(),
    "mean_candidate_type_count": validation_results["candidate_type_count"].mean(),
    "mean_reference_count_used": validation_results["reference_count_used"].mean(),
}])

print(overall.T)
print()
print(type_summary.head(20))

validation_results.to_parquet(RESULT_PARQUET, index=False)
validation_results.to_csv(RESULT_CSV, index=False)
type_summary.to_csv(TYPE_SUMMARY_CSV, index=False)

print("[PASS] validation outputs exported")

                                   0
validation_sample           1000.000
top1_match_rate                0.313
median_actual_type_rank        3.000
actual_type_available_rate     0.999
mean_candidate_type_count     33.000
mean_reference_count_used     25.000

                                   actual_bauwerksart  validation_count  \
20                                      Plattenbrücke               272   
19              Plattenbalkenbrücke, Trägerrostbrücke               261   
7                     Brücke als geschlossener Rahmen               113   
8                           Brücke als offener Rahmen               105   
0          Balkenbrücke / Mittelträger / Trapezplatte                57   
17                                   Hohlkastenbrücke                48   
23                  Rohr als Brücke, ohne Ummantelung                43   
16                 Gewölbe-/Bogenbrücke ohne Aufbeton                26   
9             Brücke mit Balken- / Plattenmischsystem            

## 07 — Validation interpretation gate

The engine is considered operationally validated only in the limited sense that:

- self-reference is excluded;
- the validation sample is reproducible;
- actual Bauwerksart can be compared with the engine's output;
- reference and candidate counts are recorded;
- no model is retrained.

A low type-match rate would **not** justify changing weights automatically.

It would be evidence that the transparent rule-based engine needs further engineering review, additional reference features, or a separately justified calibration study.

In [9]:
validation_checks = {
    "sample_size": len(validation_results) == VALIDATION_SAMPLE_SIZE,
    "self_reference_excluded": True,
    "results_finite_or_missing": np.isfinite(
        validation_results["top1_score"].dropna()
    ).all(),
    "outputs_parquet": RESULT_PARQUET.exists(),
    "outputs_csv": RESULT_CSV.exists(),
    "type_summary": TYPE_SUMMARY_CSV.exists(),
}

print("NOTEBOOK 19 VALIDATION GATE")
for name, passed in validation_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(validation_checks.values()):
    raise RuntimeError("Notebook 19 validation gate failed.")

print("[PASS] validation protocol")

NOTEBOOK 19 VALIDATION GATE
[PASS] sample_size
[PASS] self_reference_excluded
[PASS] results_finite_or_missing
[PASS] outputs_parquet
[PASS] outputs_csv
[PASS] type_summary
[PASS] validation protocol


In [10]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "notebook": "19_Plan_B_Independent_Validation_and_Decision_Engine",
    "validation_design": "held_out_existing_bridges_as_pseudo_proposed_bridges",
    "random_seed": RANDOM_SEED,
    "validation_sample_size": VALIDATION_SAMPLE_SIZE,
    "top_k_references": TOP_K_REFERENCES,
    "min_references_per_type": MIN_REFERENCES_PER_TYPE,
    "weights_base": BASE_WEIGHTS,
    "weights_validation_active": ACTIVE_WEIGHTS,
    "distance_scale_km": DISTANCE_SCALE_KM,
    "self_reference_excluded": True,
    "frozen_condition_model_changed": False,
    "frozen_condition_model_retrained": False,
    "fem_performed": False,
    "outputs": {
        "validation_results_parquet": str(RESULT_PARQUET),
        "validation_results_csv": str(RESULT_CSV),
        "type_summary_csv": str(TYPE_SUMMARY_CSV),
    },
    "input_sha256": {
        "map17": sha256_file(MAP17_INPUT),
        "library18": sha256_file(LIBRARY18_INPUT),
    },
}

MANIFEST_JSON.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("[PASS] manifest:", MANIFEST_JSON)

[PASS] manifest: C:\Datenanalyse\final Project\Output_PlanA-B\19_Plan_B_Independent_Validation\19_plan_b_validation_manifest.json


## 08 — Final status

Notebook 19 does not alter Plan A, the frozen model, or Notebook 18.

If this notebook completes, the next stage is the actual **Plan B user-facing scenario engine**:

```text
User input
    ↓
Reference cohort
    ↓
Similarity
    ↓
Performance / future-condition evidence
    ↓
Bauwerksart alternatives
    ↓
Transparent decision report
    ↓
Map / interface
```

In [11]:
final_checks = {
    "validation_results_rows": len(validation_results) == VALIDATION_SAMPLE_SIZE,
    "self_reference_excluded": True,
    "result_parquet_exists": RESULT_PARQUET.exists(),
    "result_csv_exists": RESULT_CSV.exists(),
    "type_summary_exists": TYPE_SUMMARY_CSV.exists(),
    "manifest_exists": MANIFEST_JSON.exists(),
}

print("FINAL NOTEBOOK 19 GATE")
for name, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(final_checks.values()):
    raise RuntimeError("Notebook 19 final gate failed.")

print()
print("19 STATUS: COMPLETE")
print("Validation results:", RESULT_PARQUET)

FINAL NOTEBOOK 19 GATE
[PASS] validation_results_rows
[PASS] self_reference_excluded
[PASS] result_parquet_exists
[PASS] result_csv_exists
[PASS] type_summary_exists
[PASS] manifest_exists

19 STATUS: COMPLETE
Validation results: C:\Datenanalyse\final Project\Output_PlanA-B\19_Plan_B_Independent_Validation\plan_b_validation_results.parquet
